# Standard Course Q&A RAG System
On Kaggle turn on Internet in the notebook settings, and add the course_materials folder with Add Data. On Colab upload the folder to the Files panel.

## 1. Install libraries

In [1]:
!pip install -q chromadb sentence-transformers pypdf python-docx groq gradio

print('installed')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
installed


## 2. Imports

In [2]:
import os
import re
import glob
import time
import getpass
import pandas as pd
import chromadb
import gradio as gr
from pypdf import PdfReader
from docx import Document
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder

print('imports done')

imports done


## 3. Settings

In [11]:
chunk_size = 700
chunk_overlap = 25
candidates = 12
top_k = 4
embed_model_name = 'BAAI/bge-base-en-v1.5'
rerank_model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
llm_model = 'openai/gpt-oss-120b'
# bge models need this prefix on the question only, not on the documents
query_prefix = 'Represent this sentence for searching relevant passages: '
course_names = {'CS101': 'CS101 Introduction to Python Programming', 'CS230': 'CS230 Machine Learning Fundamentals', 'CS220': 'CS220 Databases and SQL'}

print(embed_model_name, rerank_model_name, llm_model)

BAAI/bge-base-en-v1.5 cross-encoder/ms-marco-MiniLM-L-6-v2 openai/gpt-oss-120b


## 4. Find the course files

In [4]:
data_folder = None
for folder in ['course_materials', '/content/course_materials', '/kaggle/input']:
    if os.path.exists(folder):
        data_folder = folder
        break

file_paths = sorted(glob.glob(data_folder + '/**/*.*', recursive=True))
file_paths = [p for p in file_paths if p.lower().endswith(('.pdf', '.docx', '.txt', '.csv'))]

print(data_folder, len(file_paths))
for path in file_paths:
    print(path)

/kaggle/input 12
/kaggle/input/datasets/shrouqmansy/course-materials1/CS101_assessments.csv
/kaggle/input/datasets/shrouqmansy/course-materials1/CS101_faq.txt
/kaggle/input/datasets/shrouqmansy/course-materials1/CS101_lecture_notes.docx
/kaggle/input/datasets/shrouqmansy/course-materials1/CS101_syllabus.pdf
/kaggle/input/datasets/shrouqmansy/course-materials1/CS220_employees_sample.csv
/kaggle/input/datasets/shrouqmansy/course-materials1/CS220_lab3_joins.txt
/kaggle/input/datasets/shrouqmansy/course-materials1/CS220_lecture_notes.docx
/kaggle/input/datasets/shrouqmansy/course-materials1/CS220_syllabus.pdf
/kaggle/input/datasets/shrouqmansy/course-materials1/CS230_lecture_notes.docx
/kaggle/input/datasets/shrouqmansy/course-materials1/CS230_midterm_review.txt
/kaggle/input/datasets/shrouqmansy/course-materials1/CS230_syllabus.pdf
/kaggle/input/datasets/shrouqmansy/course-materials1/CS230_weekly_schedule.csv


## 5. Text extraction and cleaning

In [5]:
def read_pdf(path):
    reader = PdfReader(path)
    return '\n\n'.join(page.extract_text() or '' for page in reader.pages)


def read_docx(path):
    doc = Document(path)
    return '\n\n'.join(p.text for p in doc.paragraphs if p.text.strip())


def read_txt(path):
    with open(path, encoding='utf-8') as file:
        return file.read()


def read_csv(path):
    df = pd.read_csv(path, dtype=str)
    rows = []
    for _, row in df.iterrows():
        rows.append('; '.join(f'{col}: {row[col]}' for col in df.columns if pd.notna(row[col])) + '.')
    return '\n\n'.join(rows)


def clean_text(text):
    text = text.replace('\r', '')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


readers = {'pdf': read_pdf, 'docx': read_docx, 'txt': read_txt, 'csv': read_csv}

print(list(readers))

['pdf', 'docx', 'txt', 'csv']


## 6. Load all documents

In [6]:
documents = []
for path in file_paths:
    extension = path.split('.')[-1].lower()
    file_name = os.path.basename(path)
    course_code = file_name.split('_')[0]
    text = clean_text(readers[extension](path))
    documents.append({'course': course_names.get(course_code, course_code), 'source': file_name, 'type': extension, 'text': text})

print(pd.Series([d['type'] for d in documents]).value_counts())
print(sum(len(d['text']) for d in documents))

csv     3
txt     3
docx    3
pdf     3
Name: count, dtype: int64
37020


## 7. Chunking

In [7]:
def split_text(text, size, overlap):
    pieces = []
    for block in text.split('\n\n'):
        block = block.strip()
        if len(block) <= size:
            pieces.append(block)
        else:
            pieces += re.split(r'(?<=[.!?])\s+|\n', block)
    pieces = [p for p in pieces if p.strip()]
    chunks = []
    current = ''
    for piece in pieces:
        if current and len(current) + len(piece) > size:
            chunks.append(current)
            current = ' '.join(current.split()[-overlap:]) + '\n' + piece
        else:
            current = current + '\n' + piece if current else piece
    if current:
        chunks.append(current)
    return chunks


chunks = []
for doc in documents:
    for number, piece in enumerate(split_text(doc['text'], chunk_size, chunk_overlap)):
        header = f'Course: {doc["course"]}\nSource: {doc["source"]}\n'
        chunks.append({'id': f'{doc["source"]}_{number}', 'text': header + piece, 'course': doc['course'], 'source': doc['source']})

print(len(chunks))
print(chunks[5]['text'])

81
Course: CS101 Introduction to Python Programming
Source: CS101_faq.txt
of delay costs 10% of the assignment mark, for up to 3 days. After 3 days the mark is zero. Quizzes cannot be taken late.
Q: Can I get an extension?
A: Extensions are given only for medical reasons or emergencies with documents. Email Dr. Karim Hassan before the deadline, not after. The teaching assistant, Eng. Nour Adel, cannot approve extensions.
Q: How much attendance is required?
A: At least 75% of lectures and labs. A student below 75% cannot sit the final exam.
Q: When are the office hours?
A: Sunday and Tuesday from 12:00 to 14:00 in Room B-214. Emails are answered within 2 working days. Do not send code in screenshots; attach the file.


## 8. Embeddings and vector database

In [8]:
embedder = SentenceTransformer(embed_model_name)
texts = [c['text'] for c in chunks]
embeddings = embedder.encode(texts, batch_size=32, normalize_embeddings=True)

client = chromadb.PersistentClient(path='chroma_db')
try:
    client.delete_collection('course_materials')
except Exception:
    pass
collection = client.create_collection('course_materials')
collection.add(ids=[c['id'] for c in chunks], documents=texts, embeddings=embeddings.tolist(), metadatas=[{'course': c['course'], 'source': c['source']} for c in chunks])

print(collection.count(), embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

81 (81, 768)


## 9. Retriever with reranking

In [9]:
reranker = CrossEncoder(rerank_model_name)


def retrieve(question, course=None):
    query_vector = embedder.encode(query_prefix + question, normalize_embeddings=True).tolist()
    where = {'course': course} if course else None
    result = collection.query(query_embeddings=[query_vector], n_results=candidates, where=where)
    docs = result['documents'][0]
    metas = result['metadatas'][0]
    scores = reranker.predict([[question, doc] for doc in docs])
    ranked = sorted(zip(scores, docs, metas), key=lambda x: x[0], reverse=True)
    return ranked[:top_k]


hits = retrieve('What is the passing grade in CS101?')
for score, doc, meta in hits:
    print(round(float(score), 2), meta['source'])

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

4.12 CS101_syllabus.pdf
4.01 CS101_faq.txt
-1.11 CS101_syllabus.pdf
-1.36 CS101_assessments.csv


## 10. Connect the LLM

In [19]:
llm = Groq(api_key=getpass.getpass('Groq API key: '))

system_prompt = 'You are a teaching assistant for university courses. Answer the student question using only the context given. If the answer is not in the context, reply exactly: I could not find the answer in the course materials. Keep the answer short and clear. If the courses give different answers, say which course each answer belongs to. Do not use outside knowledge.'


def answer_question(question, course=None):
    hits = retrieve(question, course)
    context = '\n\n'.join(f'[{i + 1}] {doc}' for i, (score, doc, meta) in enumerate(hits))
    response = llm.chat.completions.create(model=llm_model, messages=[{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'}], temperature=0, max_tokens=2000)
    return response.choices[0].message.content.strip(), hits


def get_sources(answer, hits):
    if 'could not find' in answer.lower():
        return []
    good = [m for s, d, m in hits if s > 0]
    if not good and hits:
        good = [hits[0][2]]
    return sorted(set(f'{m["course"]} / {m["source"]}' for m in good))


answer, hits = answer_question('What is the late submission policy in CS101?')

print(answer)
print(get_sources(answer, hits))

Groq API key:  ········


In CS101, late assignments lose 10 % of the mark for each day they are submitted late, up to 3 days; after 3 days the mark is zero. Quizzes cannot be taken late.
['CS101 Introduction to Python Programming / CS101_faq.txt', 'CS101 Introduction to Python Programming / CS101_syllabus.pdf', 'CS220 Databases and SQL / CS220_syllabus.pdf', 'CS230 Machine Learning Fundamentals / CS230_syllabus.pdf']


## 11. Evaluate the system

In [20]:
def normalize(text):
    text = text.lower().replace('**', '').replace(',', '')
    return re.sub(r'\s+', '', text)


tests = [
    ('What is the late submission policy in CS101?', ['CS101_syllabus.pdf', 'CS101_faq.txt'], ['10%']),
    ('How is the final grade calculated in the Machine Learning course?', ['CS230_syllabus.pdf'], ['30%']),
    ('When is the SQL midterm exam?', ['CS220_syllabus.pdf'], ['april1', '2027-04-01', '1april']),
    ('What is the difference between WHERE and HAVING?', ['CS220_lecture_notes.docx'], ['group']),
    ('What is the formula for the F1 score?', ['CS230_lecture_notes.docx', 'CS230_midterm_review.txt'], ['precision']),
    ('Who is the teaching assistant for Databases and SQL?', ['CS220_syllabus.pdf', 'CS220_lab3_joins.txt'], ['mariam']),
    ('How do I handle FileNotFoundError in Python?', ['CS101_lecture_notes.docx'], ['try', 'except']),
    ('What is the passing grade in CS101?', ['CS101_syllabus.pdf', 'CS101_faq.txt'], ['50%']),
    ('What is the salary of Tarek Fahmy?', ['CS220_employees_sample.csv'], ['13200']),
    ('When is the CS101 final project topic due?', ['CS101_faq.txt', 'CS101_assessments.csv'], ['april22', '2027-04-22', '22april']),
    ('What is the late policy for the SQL labs?', ['CS220_syllabus.pdf', 'CS220_lab3_joins.txt'], ['24hours', '15%']),
    ('How many lab sessions can I miss in the Machine Learning course?', ['CS230_syllabus.pdf'], ['4', 'four']),
    ('What is the weather in Cairo today?', [], ['couldnotfind']),
]

rows = []
for question, expected_sources, keywords in tests:
    answer, hits = answer_question(question)
    found = [meta['source'] for score, doc, meta in hits]
    if expected_sources:
        retrieval_ok = any(s in found for s in expected_sources)
    else:
        retrieval_ok = None
    answer_ok = any(normalize(k) in normalize(answer) for k in keywords)
    rows.append({
        'question': question,
        'retrieval_ok': retrieval_ok,
        'answer_ok': answer_ok,
        'answer': answer,
    })
    time.sleep(4)

results = pd.DataFrame(rows)
retrieval_rate = results['retrieval_ok'].dropna().astype(bool).mean()

print(results[['question', 'retrieval_ok', 'answer_ok']])
print('Retrieval hit rate:', round(retrieval_rate * 100, 1), '%')
print('Answer accuracy:', round(results['answer_ok'].mean() * 100, 1), '%')

                                             question retrieval_ok  answer_ok
0        What is the late submission policy in CS101?         True       True
1   How is the final grade calculated in the Machi...         True       True
2                       When is the SQL midterm exam?         True       True
3    What is the difference between WHERE and HAVING?         True       True
4               What is the formula for the F1 score?         True       True
5   Who is the teaching assistant for Databases an...         True       True
6        How do I handle FileNotFoundError in Python?         True       True
7                 What is the passing grade in CS101?         True       True
8                  What is the salary of Tarek Fahmy?         True       True
9          When is the CS101 final project topic due?         True       True
10          What is the late policy for the SQL labs?         True       True
11  How many lab sessions can I miss in the Machin...         Tr

## 12. Gradio app

In [23]:
course_choices = ['All courses'] + sorted(set(d['course'] for d in documents))


def chat(question, course):
    if not question.strip():
        return 'Please type a question.', ''
    selected = None if course == 'All courses' else course
    answer, hits = answer_question(question, selected)
    return answer, '\n'.join(get_sources(answer, hits))


demo = gr.Interface(
    fn=chat,
    inputs=[gr.Textbox(label='Your question', lines=2),gr.Dropdown(course_choices, value='All courses', label='Course'), ], outputs=[gr.Markdown(label='Answer'),gr.Textbox(label='Sources'), ],title='Course Q&A Assistant',description='Ask a question about the course materials.',)
demo.launch(share=True)

print('app started')

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://e179396bed37b050d9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


app started
